# Molecular Property Prediction

**Objective:** Take a compound library through descriptor calculation, drug-likeness filtering (Lipinski Ro5), and PAINS/plausibility screening — the chemistry half of the hit-triage pipeline.

**Workflow:**
1. Compound Library — Inspect the synthetic compound collection
2. Molecular Descriptors — Native `CALCULATE_MOLECULAR_DESCRIPTORS` tool (RDKit on Snowpark)
3. Lipinski Rule of 5 — Filter and rank candidates
4. PAINS Screening — Native `VALIDATE_MOLECULE` UDTF
5. Lead Summary & Provenance


---
## 1. Setup & Connection


In [ ]:
from workbench_helpers import Workbench
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

wb = Workbench(database="SCIENTIFIC_WORKBENCH", warehouse="WORKBENCH_S")
print(f"Connected as: {wb.session.sql('SELECT CURRENT_USER()').collect()[0][0]}")

PROJECT_DB = "WORKBENCH_PROJECTS"
RESULTS = "SCIENTIFIC_WORKBENCH.RESULTS"


---
## 2. Compound Library

The synthetic library contains 1,000 candidate molecules across 4 drug projects.


In [ ]:
compounds = wb.session.sql(f"""
    SELECT PROJECT, COUNT(*) AS N, AVG(MW) AS AVG_MW, AVG(LOGP) AS AVG_LOGP
    FROM {PROJECT_DB}.DEMO_COMPOUNDS.COMPOUNDS
    GROUP BY PROJECT ORDER BY N DESC
""").to_pandas()
print(compounds.to_string(index=False))


---
## 3. Molecular Descriptors

Create a SMILES working table and run the platform's native descriptor tool in **table mode** — RDKit computes descriptors server-side on Snowpark.


In [ ]:
# Materialize a working set (top of each project by MW for variety)
wb.session.sql(f"""
    CREATE OR REPLACE TABLE {RESULTS}.NB_MOL_INPUT AS
    SELECT COMPOUND_ID, SMILES
    FROM {PROJECT_DB}.DEMO_COMPOUNDS.COMPOUNDS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY PROJECT ORDER BY MW DESC) <= 60
""").collect()

desc_table = f"{RESULTS}.NB_MOL_DESCRIPTORS"
try:
    wb.session.sql(f"""
        CALL SCIENTIFIC_WORKBENCH.CATALOG.CALCULATE_MOLECULAR_DESCRIPTORS(
            '{RESULTS}.NB_MOL_INPUT', 'table', '{desc_table}'
        )
    """).collect()
    print("Descriptor calculation completed.")
except Exception as e:
    print(f"Descriptor tool unavailable: {e}")
    print("Falling back to precomputed library descriptors.")


In [ ]:
desc = wb.session.sql(f"SELECT * FROM {desc_table}").to_pandas()
print(f"{len(desc)} molecules with computed descriptors")
desc.head(10)


---
## 4. Lipinski Rule of 5

A molecule passes Ro5 if: MW ≤ 500, LogP ≤ 5, HBD ≤ 5, HBA ≤ 10. We score every compound and flag violations.


In [ ]:
def ro5_columns(df):
    # Normalize column names across tool versions
    col = lambda *names: next((n for n in names if n in df.columns), names[0])
    mw, logp = col("MW"), col("LOGP")
    hbd, hba = col("HBD"), col("HBA")
    df["RO5_PASSES"] = (df[mw] <= 500) & (df[logp] <= 5) & (df[hbd] <= 5) & (df[hba] <= 10)
    df["RO5_VIOLATIONS"] = ((df[mw] > 500).astype(int) + (df[logp] > 5).astype(int)
                            + (df[hbd] > 5).astype(int) + (df[hba] > 10).astype(int))
    return mw, logp

mw_col, logp_col = ro5_columns(desc)
n_pass = int(desc["RO5_PASSES"].sum())
print(f"Lipinski Ro5: {n_pass}/{len(desc)} molecules pass ({100*n_pass/len(desc):.1f}%)")
desc[["SMILES" if "SMILES" in desc.columns else desc.columns[0], mw_col, logp_col, "RO5_PASSES", "RO5_VIOLATIONS"]].head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
passes = desc[desc["RO5_PASSES"]]
fails = desc[~desc["RO5_PASSES"]]
ax.scatter(passes[mw_col], passes[logp_col], s=14, c="#29b5e8", label=f"Ro5 pass ({len(passes)})")
ax.scatter(fails[mw_col], fails[logp_col], s=14, c="#f6a96a", label=f"Ro5 fail ({len(fails)})")
ax.axvline(500, color="#94a1b8", lw=0.8, ls="--")
ax.axhline(5, color="#94a1b8", lw=0.8, ls="--")
ax.set_xlabel("Molecular weight (Da)"); ax.set_ylabel("LogP")
ax.set_title("Drug-likeness: MW vs LogP (Ro5 boundaries dashed)")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()


---
## 5. PAINS Screening

The platform's `VALIDATE_MOLECULE` table function runs RDKit sanitization + PAINS alerts on individual SMILES. We screen the top Ro5-passing candidates.


In [ ]:
smiles_col = "SMILES" if "SMILES" in desc.columns else desc.columns[0]
top = desc[desc["RO5_PASSES"]].nsmallest(10, "RO5_VIOLATIONS").head(5)
results = []
for smi in top[smiles_col].tolist():
    try:
        row = wb.session.sql(f"""
            SELECT * FROM TABLE(SCIENTIFIC_WORKBENCH.CATALOG.VALIDATE_MOLECULE('{smi}'))
        """).to_pandas()
        results.append(row.iloc[0].to_dict())
    except Exception as e:
        print(f"Validation failed for {smi[:40]}...: {e}")

if results:
    screen = pd.DataFrame(results)
    show_cols = [c for c in ["smiles", "valid", "alerts", "lipinski_violations", "confidence_tier"] if c in screen.columns]
    screen[show_cols]
else:
    print("No screening results (tool unavailable).")


---
## 6. Lead Summary & Provenance

- Descriptor calculation and PAINS screening ran as governed platform tools (Snowpark + RDKit), not local code
- Ro5 filters triage the library; validated candidates feed downstream docking workflows
- All outputs persist in `SCIENTIFIC_WORKBENCH.RESULTS`


In [ ]:
wb.session.sql(f"""
    INSERT INTO SCIENTIFIC_WORKBENCH.PROVENANCE.PROVENANCE_LOG
        (action_type, target, parameters, result_summary)
    SELECT 'notebook_execution', 'molecular_properties',
           OBJECT_CONSTRUCT('library', 'DEMO_COMPOUNDS', 'n_screened', {len(desc)}),
           'Descriptors + Ro5 + PAINS screening'
""").collect()
print("Provenance logged.")
